# Evaluation CamemBERT - Format comparable SpaCy/GliNER

Pipeline :
- Stage 1 : camembert-base entraine sur 8670 docs distant supervision
- Stage 2 : fine-tuning sur 46 docs gold (splits/train.json, 1973-1993)

Evaluation sur les memes test sets que SpaCy et GliNER :
- `splits/test_before_2000.json` (30 docs, 1973-1993)
- `splits/test_after_2000.json` (50 docs, 2015-2020)

Le gold est deja dans ces fichiers (champ 'entites').
Meme fonction evaluate() que SpaCy/GliNER -> resultats directement comparables.

In [ ]:
!pip install transformers prettytable -q

import os, json, shutil
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from collections import defaultdict
from prettytable import PrettyTable

print(f'GPU : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'     {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE = '/content/drive/MyDrive/archelec_ner/'

# Modele (Stage 2 si dispo, sinon Stage 1)
MODEL_DIR = '/content/stage2_model'
if not os.path.exists(MODEL_DIR):
    if os.path.exists(DRIVE + 'stage2_model'):
        shutil.copytree(DRIVE + 'stage2_model', MODEL_DIR)
        print('OK stage2_model charge')
    elif os.path.exists(DRIVE + 'stage1_model'):
        MODEL_DIR = '/content/stage1_model'
        if not os.path.exists(MODEL_DIR):
            shutil.copytree(DRIVE + 'stage1_model', MODEL_DIR)
        print('OK stage1_model charge (stage2 absent)')
    else:
        print('ERREUR : aucun modele trouve sur Drive')
else:
    print('OK stage2_model deja present')

# Fichiers splits (texte + gold entites integres)
for fname in ['test_before_2000.json', 'test_after_2000.json', 'label2id.json']:
    src = DRIVE + 'splits/' + fname
    dst = '/content/' + fname
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'OK {fname}')
    else:
        print(f'MANQUANT {fname} -- verifie Drive/archelec_ner/splits/')

In [ ]:
with open('/content/label2id.json') as f:
    LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print('Chargement modele...')
tokenizer = AutoTokenizer.from_pretrained('camembert-base')
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f'Modele {MODEL_DIR} charge sur {device}')

In [ ]:
def predict_entities(text, max_length=512):
    """Texte brut -> liste d'entites predites via offset_mapping."""
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        max_length=max_length,
        truncation=True,
        return_tensors='pt'
    )
    offset_map = encoding['offset_mapping'][0].tolist()
    input_ids  = encoding['input_ids'].to(device)
    attn_mask  = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits

    preds  = torch.argmax(logits, dim=-1)[0].cpu().tolist()
    labels = [ID2LABEL[p] for p in preds]

    entities = []
    i = 0
    while i < len(labels):
        label = labels[i]
        if label.startswith('B-'):
            tag        = label[2:]
            start_char = offset_map[i][0]
            end_char   = offset_map[i][1]
            i += 1
            while i < len(labels) and labels[i] == 'I-' + tag:
                end_char = offset_map[i][1]
                i += 1
            entity_text = text[start_char:end_char].strip()
            if entity_text:
                entities.append({'texte': entity_text, 'tag': tag})
        else:
            i += 1
    return entities

print('predict_entities definie')
test_txt = 'Paul Dupont, candidat du Parti Socialiste dans le departement de la Seine'
print('Test:', predict_entities(test_txt))

In [ ]:
# Identique a scripts/evaluation.py
def compute_metrics_ner(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate(data, gold_by_id):
    tp_per_type         = defaultdict(int)
    fp_per_type         = defaultdict(int)
    fn_per_type         = defaultdict(int)
    support_per_type    = defaultdict(int)
    partial_tp_per_type = defaultdict(int)

    for doc in data:
        gold_doc  = gold_by_id.get(doc['id'], {})
        gold_ents = set(
            (e['texte'].strip().lower(), e['tag'].strip().upper())
            for e in gold_doc.get('entites', [])
        )
        pred_ents = set(
            (e['texte'].strip().lower(), e['tag'].strip().upper())
            for e in doc.get('predicted_entities', [])
        )
        for _, tag in gold_ents:
            support_per_type[tag] += 1
        for label in set(t for _, t in gold_ents | pred_ents):
            gl = {e for e in gold_ents if e[1] == label}
            pl = {e for e in pred_ents if e[1] == label}
            tp_per_type[label] += len(gl & pl)
            fp_per_type[label] += len(pl - gl)
            fn_per_type[label] += len(gl - pl)
        for pt, ptag in pred_ents:
            if (pt, ptag) not in gold_ents:
                for gt, gtag in gold_ents:
                    if ptag == gtag and (pt in gt or gt in pt):
                        partial_tp_per_type[ptag] += 1
                        break

    ttp = sum(tp_per_type.values())
    tfp = sum(fp_per_type.values())
    tfn = sum(fn_per_type.values())
    gp, gr, gf1 = compute_metrics_ner(ttp, tfp, tfn)

    tpart     = sum(partial_tp_per_type.values())
    tot_pred  = ttp + tfp
    tot_gold  = ttp + tfn
    pp  = (ttp + tpart) / tot_pred if tot_pred > 0 else 0.0
    pr  = (ttp + tpart) / tot_gold if tot_gold > 0 else 0.0
    pf1 = 2 * pp * pr / (pp + pr) if (pp + pr) > 0 else 0.0

    print('=' * 45)
    print('Global NER Performance (Exact Match)')
    print('=' * 45)
    t = PrettyTable(['Metric', 'Value'])
    t.add_row(['Precision', f'{gp:.4f}'])
    t.add_row(['Recall',    f'{gr:.4f}'])
    t.add_row(['F1-Score',  f'{gf1:.4f}'])
    print(t)

    print('\nGlobal NER Performance (Partial Match)')
    t2 = PrettyTable(['Metric', 'Value'])
    t2.add_row(['Precision', f'{pp:.4f}'])
    t2.add_row(['Recall',    f'{pr:.4f}'])
    t2.add_row(['F1-Score',  f'{pf1:.4f}'])
    print(t2)

    print('\nPerformance by Tag - Exact Match')
    t3 = PrettyTable(['Tag', 'Precision', 'Recall', 'F1-Score', 'Support', 'Partial TP'])
    for label in sorted(set(list(tp_per_type) + list(support_per_type))):
        p, r, f = compute_metrics_ner(tp_per_type[label], fp_per_type[label], fn_per_type[label])
        t3.add_row([label, f'{p:.4f}', f'{r:.4f}', f'{f:.4f}',
                    support_per_type[label], partial_tp_per_type[label]])
    print(t3)

    all_labels = sorted(set(list(tp_per_type) + list(support_per_type)))
    return {
        'exact':   {'precision': round(gp, 3),  'recall': round(gr, 3),  'f1': round(gf1, 3)},
        'partial': {'precision': round(pp, 3),  'recall': round(pr, 3),  'f1': round(pf1, 3)},
        'per_type': {
            l: {
                'precision':  round(compute_metrics_ner(tp_per_type[l], fp_per_type[l], fn_per_type[l])[0], 3),
                'recall':     round(compute_metrics_ner(tp_per_type[l], fp_per_type[l], fn_per_type[l])[1], 3),
                'f1':         round(compute_metrics_ner(tp_per_type[l], fp_per_type[l], fn_per_type[l])[2], 3),
                'support':    support_per_type[l],
                'partial_tp': partial_tp_per_type[l]
            }
            for l in all_labels
        }
    }

print('evaluate definie')

In [ ]:
# Inference sur test_before_2000 (1973-1993)
# Gold = champ 'entites' directement dans le fichier splits
with open('/content/test_before_2000.json', encoding='utf-8') as f:
    test_before = json.load(f)

gold_before = {doc['id']: doc for doc in test_before}

print(f'Inference sur {len(test_before)} docs...')
predictions_before = []
for i, doc in enumerate(test_before):
    predictions_before.append({
        'id':                  doc['id'],
        'annee':               doc['annee'],
        'predicted_entities':  predict_entities(doc['texte'])
    })
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(test_before)}')

print('\n=== RESULTATS test_before_2000 (1973-1993) ===')
results_before = evaluate(predictions_before, gold_before)

In [ ]:
# Inference sur test_after_2000 (2015-2020)
with open('/content/test_after_2000.json', encoding='utf-8') as f:
    test_after = json.load(f)

gold_after = {doc['id']: doc for doc in test_after}

print(f'Inference sur {len(test_after)} docs...')
predictions_after = []
for i, doc in enumerate(test_after):
    predictions_after.append({
        'id':                  doc['id'],
        'annee':               doc['annee'],
        'predicted_entities':  predict_entities(doc['texte'])
    })
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(test_after)}')

print('\n=== RESULTATS test_after_2000 (2015-2020) ===')
results_after = evaluate(predictions_after, gold_after)

In [ ]:
model_name = 'stage2' if 'stage2' in MODEL_DIR else 'stage1'
os.makedirs('/content/results_camembert', exist_ok=True)

f_before = f'/content/results_camembert/{model_name}_metrics_before_2000.json'
f_after  = f'/content/results_camembert/{model_name}_metrics_after_2000.json'
with open(f_before, 'w') as f: json.dump(results_before, f, indent=2)
with open(f_after,  'w') as f: json.dump(results_after,  f, indent=2)

drive_out = DRIVE + 'results/CamemBERT/'
os.makedirs(drive_out, exist_ok=True)
shutil.copy(f_before, drive_out)
shutil.copy(f_after,  drive_out)
print(f'Resultats sauvegardes -> Drive/archelec_ner/results/CamemBERT/')

In [ ]:
def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f'ABSENT : {path}')
    return None

spacy_b  = load_json(DRIVE + 'results/Spacy/trained/trained_metrics_before_2000.json')
spacy_a  = load_json(DRIVE + 'results/Spacy/trained/trained_metrics_after_2000.json')
gliner_b = load_json(DRIVE + 'results/Gliner/raw/metrics_before_2000_0.3.json')
gliner_a = load_json(DRIVE + 'results/Gliner/raw/metrics_after_2000_0.3.json')

print('=' * 80)
print('COMPARAISON FINALE --- SpaCy vs GliNER vs CamemBERT')
print('=' * 80)

t = PrettyTable(['Modele', 'Split', 'Prec(E)', 'Rec(E)', 'F1(E)', 'Prec(P)', 'Rec(P)', 'F1(P)'])

def add_rows(tbl, name, rb, ra):
    for split, res in [('before_2000', rb), ('after_2000', ra)]:
        if res is None:
            tbl.add_row([name, split] + ['-'] * 6)
        else:
            e, p = res['exact'], res['partial']
            tbl.add_row([name, split,
                f"{e['precision']:.3f}", f"{e['recall']:.3f}", f"{e['f1']:.3f}",
                f"{p['precision']:.3f}", f"{p['recall']:.3f}", f"{p['f1']:.3f}"])

add_rows(t, 'SpaCy trained',         spacy_b,       spacy_a)
add_rows(t, 'GliNER (t=0.3)',         gliner_b,      gliner_a)
add_rows(t, 'CamemBERT ' + model_name, results_before, results_after)
print(t)